<!--
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
-->

# GÜN 061: enum class (Kapsüllenmiş İsim Alanı, Güçlü Tip Güvenliği, Belirlenmiş Altyapı Tipi, C++20 using enum ve Donanım Register Bitmask Mimarisi)

[![License: All Rights Reserved](https://img.shields.io/badge/license-All%20Rights%20Reserved-red?style=flat-square)](https://github.com/seydivakkas)
[![C++ Standard](https://img.shields.io/badge/C%2B%2B-20-blue?style=flat-square&logo=c%2B%2B)](https://en.cppreference.com/w/cpp/20)
[![Domain](https://img.shields.io/badge/Domain-Avionics%20Flight%20Control%20%26%20Hardware%20Registers-darkgreen?style=flat-square)]()
[![Safety](https://img.shields.io/badge/Safety-MISRA%20C%2B%2B%3A2023%20%7C%20DO--178C%20DAL--A-orange?style=flat-square)]()

---

## 🏛️ Akademik ve Kurumsal Bildirim
Bu modül, **ASELSAN, HAVELSAN, ROKETSAN, TUSAŞ ve STM** gibi savunma sanayii kuruluşlarının görev-kritik aviyonik uçuş kontrol, silah yönetim ve radar alt sistemlerinde kullanılan durum makineleri ve donanım kayıtçısı (register) arayüzlerinde tip güvenliğini garantiye almak için tasarlanmıştır.

- **Müfredat Fazı:** Faz 1 (F1) — Modern C++20 Çekirdek Sistem Programlama
- **Ön Koşul Konu:** Gün 060 (`std::variant` ve type-safe union)
- **Pedagojik Derinlik:** 40-Nokta Tam Kapsam Doğrulaması (First Principles -> Donanım Bellek Anatomisi -> Sabotaj -> Benchmarking -> Sektörel Standartlar)


## 1. 🎯 Bugünkü Problem: Eski C-Enum Kapsam Kirliliği ve Örtülü Tamsayı Felaketi

Geleneksel C ve eski C++98 standartlarında kullanılan klasik numaralandırmalar (`enum`), sistem mühendisliğinde üç büyük felakete zemin hazırlar:

### 1. Global İsim Alanı Kirliliği (Global Scope Pollution & Name Collisions)
Eski C enum'larının üyeleri doğrudan enum'ın tanımlandığı kapsama sızar:
```cpp
// ESKİ VE TEHLİKELİ C-ENUM:
enum RadarState   { IDLE, TRACKING, ERROR };
enum ThermalState { IDLE, CALIBRATING, ERROR }; // DERLEME HATASI! IDLE ve ERROR zaten tanımlı!
```
Yazılımcılar bu çakışmayı önlemek için `RADAR_IDLE`, `THERMAL_IDLE` gibi çirkin ön ekler (prefix) uydurmak zorunda kalırdı.

### 2. Örtülü Tamsayıya Dönüşüm (Implicit Conversion to `int`)
Eski enum'lar örtülü olarak `int` türüne dönüşür:
```cpp
RadarState r = TRACKING;      // r = 1
ThermalState t = CALIBRATING;  // t = 1

if (r == t) {
    // DERLENİR VE ÇALIŞIR! İkisi de 1 olduğu için radar ile termal eşit kabul edilir!
    // Aviyonik sistemde iki bağımsız alt sistem birbirine karışır!
}

int velocity = r + 42; // Anlamsız aritmetik serbesttir!
```

### 3. Altyapı Tipi Belirsizliği (Underlying Type Indeterminacy)
Eski C enum'larının boyutu derleyiciye bağlıdır (*implementation-defined*). Bir derleyici `sizeof(enum)` için 1 bayt seçerken, diğeri 4 bayt seçebilir. Bu durum, MIL-STD-1553, CAN Bus veya SPI protokollerinde donanım paketlerinin bit kaymasına ve çökmesine yol açar!

---

### Bugünkü Çözüm: `enum class` (Scoped Enumerations)
C++11 ile gelen ve C++20'de `using enum` ile mükemmelleştirilen **`enum class`**:
- **Kapsüllenmiştir:** Değerler `RadarState::Idle` şeklinde ad alanında korunur; isim çakışması imkansızdır.
- **Güçlü Tiplidir (Strongly Typed):** Asla kendiliğinden `int`'e dönüşmez; farklı enum türleri birbirleriyle karşılaştırılamaz.
- **Sabit Boyutludur:** `enum class RadarState : uint8_t` ile donanımdaki bayt boyutu derleme zamanında mühürlenir.


## 2. ❓ Bu Konuyu Neden Öğreniyoruz? (Savunma Sanayii Gerekçesi)

Savunma ve havacılık sistemlerinde `enum class` kullanımını zorunlu kılan iki ana kural vardır:

### 1. Aviyonik Durum Makinelerinde Tip Karışıklığının Önlenmesi
Uçuş kontrol bilgisayarında (FCS) otopilot modu (`AutopilotMode::AltitudeHold`) ile iniş takımı modu (`GearMode::Up`) yanlışlıkla aynı tamsayı değerine sahip olabilir (her ikisi de `1` olabilir).
- Eğer `enum class` kullanılmazsa, otopilot komutunu iniş takımı servosu yanlışlıkla çalıştırabilir.
- `enum class` kullanıldığında derleyici bu karşılaştırmayı derleme zamanında (*compile-time*) reddeder:
  `error: no match for 'operator==' (operand types are 'AutopilotMode' and 'GearMode')`.

### 2. Donanım Kayıtçıları (Hardware Registers) ve Ağ Paketleri ile Birebir Eşleşme
Savunma donanımlarında SPI, I2C veya PCIe üzerinden FPGA kayıtçılarına veri yazılırken:
```cpp
enum class PowerState : uint8_t {
    Off   = 0x00,
    Sleep = 0x01,
    Ready = 0x02,
    Armed = 0x0F
};
static_assert(sizeof(PowerState) == 1, "Donanım register boyutu 1 bayt olmalıdır!");
```
Belirlenmiş altyapı tipi (`fixed underlying type`), donanım sürücüsünün FPGA ile bit düzeyinde hatasız haberleşmesini garanti eder.


## 3. 🔍 5N1K Analizi (Sistem ve Donanım Sınırları)

| Soru | Sistem & Donanım Mühendisliği Yanıtı |
| :--- | :--- |
| **Ne? (What)** | **`enum class` (Scoped Enumeration)**, üyelerinin isim alanını kendi içinde hapseden, tamsayılara örtülü dönüşümü engelleyen ve temel veri tipini (`underlying type`) açıkça belirleyen modern C++ tür tanımlayıcısıdır. |
| **Neden? (Why)** | Global isim kirliliğini, farklı durum kodlarının birbiriyle karışmasını, tehlikeli örtülü tamsayı dönüşümlerini ve belirsiz bellek boyutunu derleme zamanında ortadan kaldırmak için. |
| **Nasıl? (How)** | `enum class Name : UnderlyingType { Value1, Value2 };` sözdizimi ile tanımlanır. Değerlere `Name::Value1` olarak erişilir. Tamsayıya dönüştürmek gerekirse açıkça `static_cast<Type>` veya C++23 `std::to_underlying` kullanılır. |
| **Nerede? (Where)** | TUSAŞ KAAN Uçuş Kontrol Bilgisayarı durum makinelerinde, ROKETSAN güdüm kilit durumlarında, ASELSAN radar PRF modlarında ve MIL-STD-1553 bus mesaj kodlarında. |
| **Ne Zaman? (When)** | Bir değişkenin sınırlı, belirli ve adlandırılmış bir durum kümesinden birini alması gerektiği her yerde; sihirli sayılar (`magic numbers`) yerine her zaman. |
| **Kim? (Who)** | Aviyonik durum makinesi tasarımcıları, donanım sürücü mimarları, gömülü yazılım mühendisleri ve MISRA denetçileri. |


## 4. 💡 Günlük Hayat Benzetmesi (Zihinsel Model / Analoji)

> ### 📻 Taktik Telsiz Frekansları ve Hava Trafik Çağrı Kodları Analojisi
> 
> Bir uluslararası askeri hava üssünün kontrol kulesini düşünün:
> 
> 1. **Eski C-Enum Yolu (Ortak Megafon / Kapsam Kirliliği):**
>    Kuledeki herkes tek bir ortak megafona konuşur. Pist sorumlusu megafondan "KIRMIZI" diye bağırır (Pist kapalı demektir). Aynı anda itfaiye sorumlusu da megafona "KIRMIZI" der (Yangın var demektir). Savunma bataryası komutanı megafona "KIRMIZI" der (Düşman füzesi yaklaşıyor, ateş serbest demektir).
>    Tüm istasyonlar tek bir kelimeyi (`KIRMIZI = 1`) duyar. Bir itfaiye tatbikatı anında hava savunma bataryası "Ateş Serbest" emri sanıp füzeleri ateşleyebilir! (**Örtülü Karışıklık ve Kapsam Çatışması**).
> 
> 2. **`enum class` Yolu (Korumalı Sayısal Taktik Frekanslar):**
>    Her birime kendi özel şifreli kanalı tahsis edilir:
>    - `PistDurumu::Kirmizi`
>    - `ItfaiyeDurumu::Kirmizi`
>    - `HavaSavunmaDurumu::Kirmizi`
>    
>    Hava savunma bataryasının telsizi sadece `HavaSavunmaDurumu` kanalındaki sinyalleri dinleyebilir. İtfaiyenin "Kırmızı" anonsu hava savunma telsizine fiziksel olarak ulaşamaz; frekanslar ve protokoller izoledir!
>    Eğer bir acemi operatör itfaiye mesajını hava savunma sistemine girmeye çalışırsa cihaz "Uyumsuz Protokol / Geçersiz Tip!" diyerek komutu derhal reddeder!


## 5. 🧱 İlk İlkeler (First Principles) ve Matematiksel / Donanımsal Model

### 5.1 Bellek Boyutu ve Hizalama Matematiği

C++20'de `enum class` tanımlanırken altyapı türü (`underlying type`) açıkça belirtilmezse, derleyici varsayılan olarak **`int`** (genellikle 4 bayt, signed) kabul eder:

```cpp
enum class DefaultState { A, B };         // sizeof == 4 Bayt (int)
enum class CompactState : uint8_t { A, B }; // sizeof == 1 Bayt (unsigned char)
enum class LargeState   : uint64_t { A }; // sizeof == 8 Bayt (unsigned long long)
```

#### Donanımsal Matematiksel Eşleme:
Belirlenmiş bir enum türünün altyapı tipi $T_{under}$ ise:

$$	ext{sizeof}(	ext{enum class}) = 	ext{sizeof}(T_{under})$$
$$	ext{alignof}(	ext{enum class}) = 	ext{alignof}(T_{under})$$

- Eğer $T_{under} = 	ext{uint8\_t}$ seçilirse, bellek tüketimi 4 bayttan 1 bayta iner (**%75 RAM tasarrufu**).
- Milyonlarca durum içeren radar temas dizilerinde veya telemetri paketlerinde bu tasarruf kritik önbellek (L1 Data Cache) sığasını doğrudan korur.

---

### 5.2 Değer Kümesi ve Tanımsız Davranış Sınırı (Value Range & UB)

Bir `enum class`'ın geçerli değer aralığı, altyapı tipinin değer aralığı ile sınırlıdır:
```cpp
enum class SensorMode : uint8_t {
    Standby = 0,
    Active  = 1,
    Calib   = 2
};
```
- Donanımdan veya ağdan gelen bozuk bir bayt (örneğin `0xFF`), `static_cast<SensorMode>(0xFF)` ile enum'a dönüştürülebilir!
- C++ standardına göre bu işlem altyapı tipi `uint8_t` olduğu için tanımsız davranış (*Undefined Behavior*) oluşturmaz; **ANCAK** enum'ın mantıksal değer kümesi dışındadır!
- **Savunma Sanayii Kuralı (MISRA C++:2023):** Dış dünyadan (CAN, seri port, ağ) alınan tamsayılar enum'a çevrilmeden önce mutlaka sınır kontrolünden (*boundary validation*) geçirilmelidir!


## 6. 📐 ASCII ve Mermaid Sistem Mimarisi Şeması

### 6.1 Bellek ve Kapsam Karşılaştırması

```text
[ ESKİ C-ENUM (Global Kapsama Sızma & 4 Bayt int Zorunluluğu) ]
Global Scope:
  IDLE = 0 (4 Bayt) ──┐
  TRACK = 1 (4 Bayt)  ├──> İsim çakışması kaçınılmaz!
  ERROR = 2 (4 Bayt) ──┘    int dönüşümü serbest!

[ MODERN C++ enum class (Kapsüllenmiş İsim Alanı & 1 Bayt uint8_t) ]
RadarState Scope:
  ┌──────────────────────────────────────────────┐
  │ RadarState::Idle     = 0x00 (1 Bayt)         │
  │ RadarState::Tracking = 0x01 (1 Bayt)         │
  │ RadarState::Error    = 0xFF (1 Bayt)         │
  └──────────────────────────────────────────────┘
  ▲ İsim alanı mühürlüdür. Örtülü int dönüşümü İMKANSIZDIR!
```

---

### 6.2 Aviyonik Uçuş Kontrol Durum Makinesi (Mermaid)

```mermaid
stateDiagram-v2
    [*] --> Disarmed : Sistem Başlatma

    Disarmed --> PreFlightCheck : Mode = ArmedRequest
    PreFlightCheck --> Standby : Test Başarılı
    PreFlightCheck --> Error : Sensör Arızası

    Standby --> ManualFlight : Mode = Manual
    Standby --> AutoPilot : Mode = AutoPilotNav

    ManualFlight --> AutoPilot : Pilot Devri
    AutoPilot --> EmergencyLanding : Mode = CriticalFault

    AutoPilot --> ReturnToBase : Mode = RTB
    ReturnToBase --> Disarmed : İniş Tamamlandı
    EmergencyLanding --> [*] : Motor Kapatma
```


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# MODÜL 1: Temel Çekirdek Mekanizma (Simulated Scoped Enum)
# C++20 enum class Güçlü Tip Kontrolü ve Kapsülleme Motoru
# ==============================================================================

from enum import Enum
from typing import Type, Any

class ScopedEnumMeta(type):
    """Modern C++ 'enum class' Güçlü Tipleme Davranışını Zorunlu Kılan Metaclass"""
    def __repr__(cls):
        return f"<enum class '{cls.__name__}'>"

class ScopedEnum(metaclass=ScopedEnumMeta):
    """
    Modern C++ enum class benzetimi:
    - Asla örtülü olarak int veya diğer enum'larla karşılaştırılamaz.
    - Kapsüllenmiş isim alanına sahiptir.
    """
    def __init__(self, name: str, value: int, underlying_bytes: int = 1):
        self._name = name
        self._value = value
        self._bytes = underlying_bytes

    @property
    def name(self) -> str: return self._name
    @property
    def value(self) -> int: return self._value
    @property
    def sizeof(self) -> int: return self._bytes

    def __repr__(self):
        return f"{self.__class__.__name__}::{self._name}"

    def __eq__(self, other: Any) -> bool:
        # Katı Tip Kontrolü: Yalnızca aynı enum class türü ile karşılaştırılabilir!
        if not isinstance(other, self.__class__):
            raise TypeError(
                f"[Derleme Hatası - MISRA C++:2023]: '{self.__class__.__name__}' ile "
                f"'{type(other).__name__}' türleri karşılaştırılamaz! Örtülü int dönüşümü yasaktır!"
            )
        return self._value == other._value

    def __hash__(self):
        return hash((self.__class__.__name__, self._value))

def run_module_01_core():
    print("[*] Modül 1: enum class Güçlü Tip Güvenliği Test Ediliyor...")

    # İki Ayrı Aviyonik Enum Tanımlayalım
    class RadarMode(ScopedEnum): pass
    RadarMode.Idle = RadarMode("Idle", 0, underlying_bytes=1)
    RadarMode.Tracking = RadarMode("Tracking", 1, underlying_bytes=1)
    RadarMode.Engagement = RadarMode("Engagement", 2, underlying_bytes=1)

    class FlirMode(ScopedEnum): pass
    FlirMode.Idle = FlirMode("Idle", 0, underlying_bytes=1)
    FlirMode.Calibrating = FlirMode("Calibrating", 1, underlying_bytes=1)

    print(f"  -> Radar Durumu: {RadarMode.Tracking} (Boyut: {RadarMode.Tracking.sizeof} Bayt)")
    print(f"  -> FLIR Durumu:  {FlirMode.Idle} (Boyut: {FlirMode.Idle.sizeof} Bayt)")

    # 1. Aynı tür içi geçerli karşılaştırma
    assert RadarMode.Tracking == RadarMode.Tracking
    assert RadarMode.Tracking != RadarMode.Idle

    # 2. Farklı türler arası karşılaştırma blokajı
    caught_type_error = False
    try:
        # RadarMode.Idle (0) == FlirMode.Idle (0) -> C'de true olurdu, C++ enum class'ta yasak!
        _ = (RadarMode.Idle == FlirMode.Idle)
    except TypeError as ex:
        caught_type_error = True
        print(f"  -> [BEKLENEN DERLEME ZAMANI ENGELİ]: {ex}")

    assert caught_type_error is True, "HATA: Farklı enum türleri birbirine karıştı!"
    print("  -> [PASS] Modül 1 çekirdek tip güvenliği mekanizması başarıyla doğrulandı.")

if __name__ == "__main__":
    run_module_01_core()


## 8. Modül 2: Donanıma Yakın / İleri Düzey Alternatif Mekanizma: C++20 `using enum` ve Tip Güvenli Bitmask

### 8.1 C++20 `using enum` Kolaylığı
`enum class`'ın tek ergonomik dezavantajı, her seferinde `RadarMode::` gibi uzun ad alanını yazma zorunluluğudur.
C++20, bu sorunu yerel kapsamda **`using enum`** bildirimiyle çözmüştür:

```cpp
void handleRadar(RadarMode mode) {
    using enum RadarMode; // C++20: Yalnızca bu fonksiyon içinde Idle, Tracking doğrudan erişilebilir!
    switch (mode) {
        case Idle:       standby(); break;
        case Tracking:   trackTarget(); break;
        case Engagement: fireInterceptor(); break;
    }
}
```
Global kapsam kirlenmez, sadece fonksiyon içinde temiz ve zarif kod yazılır.

---

### 8.2 Donanım Register'ları İçin Tip Güvenli Bit Bayrakları (Bitmask Operators)
Gömülü sistemlerde donanım register'larında birden fazla bayrak `OR` (`|`) ile birleştirilir.
Normalde `enum class` ile bitwise işlem yapıldığında derleyici tamsayıya örtülü dönüşümü engeller.
Modern C++'ta bunu aşmak için **tip güvenli operatör aşırı yüklemeleri (Operator Overloading)** yazılır:

```cpp
enum class SensorFlags : uint8_t {
    None         = 0x00,
    GpsLocked    = 0x01,
    ImuAligned   = 0x02,
    RadarActive  = 0x04,
    LaserArmed   = 0x08
};

// Tip Güvenli Bitwise OR Operatörü:
constexpr SensorFlags operator|(SensorFlags a, SensorFlags b) noexcept {
    return static_cast<SensorFlags>(
        static_cast<uint8_t>(a) | static_cast<uint8_t>(b)
    );
}

// Kullanım:
SensorFlags activeState = SensorFlags::GpsLocked | SensorFlags::RadarActive; // Tamamen tip güvenli!
```
Bu sayede donanım kayıtçısına yazarken rastgele tamsayılar yazılamaz; yalnızca geçerli `SensorFlags` kombinasyonları yazılabilir!


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# MODÜL 2: Tip Güvenli Bitmask ve Donanım Register Kontrolörü
# Bitwise Bayraklar ve C++20 using enum Simülasyonu
# ==============================================================================

class HardwareRegister:
    """FPGA/ASIC Bellek Eşlemeli Kayıtçı (MMIO) Simülatörü"""
    def __init__(self, address: int):
        self.address = address
        self.raw_value: int = 0x00

    def write_flags(self, flags: Any) -> None:
        if not hasattr(flags, "underlying_mask"):
            raise TypeError("Sadece tip güvenli BitmaskFlags yazılabilir!")
        self.raw_value = flags.underlying_mask

    def read_flags(self) -> int:
        return self.raw_value

class BitmaskFlags:
    def __init__(self, mask: int = 0):
        self.underlying_mask = mask & 0xFF  # 1 Baytlık register

    def __or__(self, other: 'BitmaskFlags') -> 'BitmaskFlags':
        return BitmaskFlags(self.underlying_mask | other.underlying_mask)

    def __and__(self, other: 'BitmaskFlags') -> 'BitmaskFlags':
        return BitmaskFlags(self.underlying_mask & other.underlying_mask)

    def contains(self, flag: 'BitmaskFlags') -> bool:
        return (self.underlying_mask & flag.underlying_mask) == flag.underlying_mask

    def __repr__(self):
        return f"Bitmask(0x{self.underlying_mask:02X})"

# Aviyonik Sensör Donanım Bayrakları (uint8_t)
class SystemStatus:
    GPS_SYNC   = BitmaskFlags(0x01) # Bit 0
    IMU_READY  = BitmaskFlags(0x02) # Bit 1
    RADAR_ON   = BitmaskFlags(0x04) # Bit 2
    IFF_SECURE = BitmaskFlags(0x08) # Bit 3

def run_module_02_bitmask():
    print("\n[*] Modül 2: Tip Güvenli Bitmask ve Donanım Register Kontrolü...")

    reg = HardwareRegister(address=0x4000_1000)

    # 1. GPS + IMU + RADAR Bayraklarını Birleştirelim:
    active_config = SystemStatus.GPS_SYNC | SystemStatus.IMU_READY | SystemStatus.RADAR_ON
    reg.write_flags(active_config)

    print(f"  -> Register'a Yazılan Bayrak Değeri: {active_config} -> Ham Değer: 0x{reg.read_flags():02X}")
    assert reg.read_flags() == 0x07  # 0x01 | 0x02 | 0x04 = 0x07

    # 2. Durum Sorgulama:
    assert active_config.contains(SystemStatus.GPS_SYNC) is True
    assert active_config.contains(SystemStatus.IFF_SECURE) is False
    print("  -> GPS ve IMU aktifliği doğrulandı, IFF pasif olarak tespit edildi.")
    print("  -> [PASS] Tip güvenli donanım register bitmask mekanizması onaylandı.")

if __name__ == "__main__":
    run_module_02_bitmask()


## 10. Modül 3: Dinamik Sınır Durumlar, Geri Basınç ve Sınır Dışı Enum Değeri Enjeksiyonu

Aviyonik seri hatlarından (RS-422, MIL-STD-1553 veya CAN Bus) baytlar okunurken en sinsi hata kaynağı:
**"Enum Değer Kümesi Dışında Kalan Baytlar"** dır.

### Sınır Durum Analizi:
```cpp
enum class MissileCommand : uint8_t {
    Hold       = 0x01,
    Arm        = 0x02,
    Launch     = 0x03,
    SelfDestruct = 0x04
};
```
1. **İletişim Hattı Gürültüsü (Line Noise):** Veri yolunda 1 bit kayarsa ve gelen bayt `0x05` olursa ne olur?
2. `static_cast<MissileCommand>(0x05)` çağrısı derleme hatası vermez; `MissileCommand` değişkeni `0x05` değerini alır!
3. Eğer `switch(cmd)` yapısında `default:` dalı yazılmamışsa veya derleyicinin eksik dal uyarısı (`-Wswitch`) atlanmışsa, sistem bilinmeyen bir ara duruma düşer.
4. **Savunma Mimarisi (Boundary Validator):**
   Gelen bayt enum'a cast edilmeden önce `isValidCommand(byte)` doğrulamasından geçmeli; geçersiz değerler derhal **Fast-Fail** ile telemetri arıza loguna yazılmalıdır!


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# MODÜL 3: Dayanıklı Aviyonik Komut Ayrıştırıcı ve Sınır Doğrulayıcı
# Gürültülü Veri Yolu Simülasyonu ve Fast-Fail Validasyonu
# ==============================================================================

class MissileCommand:
    VALID_CODES = {0x01: "HOLD", 0x02: "ARM", 0x03: "LAUNCH", 0x04: "ABORT"}

    def __init__(self, code: int):
        if code not in self.VALID_CODES:
            raise ValueError(f"[Geçersiz Aviyonik Enum Değeri]: 0x{code:02X} tanımlı komut kümesinde yok!")
        self.code = code
        self.label = self.VALID_CODES[code]

    def __repr__(self): return f"MissileCommand::{self.label}"

def parse_bus_packet(raw_byte: int) -> str:
    try:
        cmd = MissileCommand(raw_byte)
        return f"KOMUT_ONAYLANDI -> {cmd}"
    except ValueError as ex:
        # Fast-Fail Güvenlik Savunması
        return f"KOMUT_REDDEDİLDİ -> {ex}"

def run_module_03_parser():
    print("\n[*] Modül 3: Aviyonik Bus Komut Ayrıştırıcı ve Sınır Denetimi...")

    # Senaryo A: Geçerli Komutlar
    res1 = parse_bus_packet(0x02) # ARM
    res2 = parse_bus_packet(0x03) # LAUNCH
    print(f"  -> Paket 1: {res1}")
    print(f"  -> Paket 2: {res2}")
    assert "KOMUT_ONAYLANDI" in res1
    assert "LAUNCH" in res2

    # Senaryo B: Gürültülü / Sabote Edilmiş Hatalı Bayt (0x99)
    res_err = parse_bus_packet(0x99)
    print(f"  -> Paket 3 (Gürültü): {res_err}")
    assert "KOMUT_REDDEDİLDİ" in res_err
    print("  -> [PASS] Sınır doğrulayıcı tanımlanmamış enum değerlerini başarıyla engelledi.")

if __name__ == "__main__":
    run_module_03_parser()


## 12. Sabotaj ve Arıza Enjeksiyonu (Chaos Engineering / Bilerek Boz)

### 12.1 Sabotaj Hipotezi
Güdümlü füze yer kontrol istasyonunda iki ayrı alt sistem vardır:
- **`RadarFrequencyBand`** (0 = X_BAND, 1 = KU_BAND, 2 = KA_BAND)
- **`WarheadArmingState`** (0 = SAFE, 1 = ARMED, 2 = DETONATE)

Eski C tarzı kodda her iki enum da tamsayıya dönüştüğü için, radar anteni frekansını ayarlamakla görevli acemi bir mühendis kazara şu hatayı yapar:
```c
// KORUMASIZ C KODU:
if (radar_band == ARMED) { // FARKLI TİPLER EŞİTLİK KONTROLÜNE GİRER!
    detonateWarhead();     // Radar Ku-Bandı seçtiğinde füze fırlatma rampasında patlar!
}
```

### 12.2 Beklenen Failure Belirtisi
- **C-Enum:** Derleyici sessizce `1 == 1` kontrolünü yapar, koşul `true` döner ve savaş başlığı patlatılır (**Ölümcül Güvenlik Faciası**).
- **`enum class`:** Derleyici `operator==(RadarFrequencyBand, WarheadArmingState)` bulunamadığı için kodu derlemeyi **anında durdurur**. Hata sahaya asla çıkamaz!


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# MODÜL 4: Sabotaj Senaryosu (Chaos Engineering)
# C-Enum Tip Karışıklığı Sabotajı vs enum class Kalkanı
# ==============================================================================

def run_module_04_sabotage():
    print("\n[*] Modül 4: Sabotaj ve Arıza Enjeksiyonu Başlatılıyor...")

    # [SENARYO 1: C-STYLE ENUM ZEHİRLENMESİ VE PATLAMA]
    print("  [1] KORUMASIZ C-TARZI ENUM FELAKETİ:")
    # İki farklı kavram aynı int değerine sahiptir (1)
    C_RADAR_KU_BAND = 1
    C_WARHEAD_ARMED = 1

    # Sistem acemi kod yazımıyla kontrol ediliyor:
    if C_RADAR_KU_BAND == C_WARHEAD_ARMED:
        print("      [KRİTİK HATA]: Radar frekansı Ku-Band iken Harp Başlığı ARMED zannedildi!")
        print("      -> Savaş başlığı fırlatma rampasında zamansız tetiklendi! (FELAKET!)")

    # [SENARYO 2: enum class TİP GÜVENLİK BARİYERİ]
    print("  [2] enum class DERLEME ZAMANI TİP GÜVENLİK KALKANI:")
    class RadarBand(ScopedEnum): pass
    RadarBand.KuBand = RadarBand("KuBand", 1)

    class WarheadState(ScopedEnum): pass
    WarheadState.Armed = WarheadState("Armed", 1)

    sabotage_blocked = False
    try:
        # İki farklı enum class'ı karşılaştırmaya çalışalım:
        if RadarBand.KuBand == WarheadState.Armed:
            pass
    except TypeError as ex:
        sabotage_blocked = True
        print(f"      [SAVUNMA BAŞARILI]: {ex}")

    assert sabotage_blocked is True, "HATA: Farklı enum class'lar karşılaştırılabildi!"
    print("  -> [PASS] enum class tip ayrımı sayesinde aviyonik felaket %100 önlendi.")

if __name__ == "__main__":
    run_module_04_sabotage()


## 14. Modül 4: Ampirik Kıyaslama ve SLA / Gecikme Analizi (p50/p95/p99)

Görev-kritik radar sinyal işleme sistemlerinde durum makineleri her mikro-saniyede milyonlarca kez değerlendirilir.

### 1. `enum class` ile Switch-Case Atlama Tablosu (Jump Table)
Modern derleyiciler (GCC / Clang), sıralı değerlere sahip bir `enum class` üzerinde `switch(state)` yapısını doğrudan tek bir işlemci atlama tablosuna dönüştürür:
- İşlemci dal tahmin edicisi (*branch predictor*) zorlanmaz.
- Atlama süresi **$O(1)$** ve tam deterministiktir (~1.2 nanosaniye).

### 2. Dize Tabanlı (String) veya Polimorfik Durum Kontrolü
Durumları string (`"TRACKING"`) veya miras sınıfları ile kontrol etmek:
- Her döngüde `strcmp` veya vtable araması yapar.
- Determinizm bozulur, gecikme dalgalanması (*jitter*) artar.


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# MODÜL 5: Kapsamlı Ampirik Kıyaslama Motoru (Benchmark)
# 100.000 Durum Değerlendirmesinde: enum class vs String Durum Kontrolü
# ==============================================================================

import time

def run_module_05_benchmark():
    print("\n[*] Modül 5: enum class (Tamsayı) vs String Durum Değerlendirme Ölçümü...")
    iterations = 100_000

    # 1. enum class / Integer Dispatch Benzetimi
    start_enum = time.perf_counter()
    state_code = 2
    for i in range(iterations):
        # O(1) integer switch benzetimi:
        if state_code == 0: val = 10
        elif state_code == 1: val = 20
        elif state_code == 2: val = 30
        else: val = 0
    t_enum_ms = (time.perf_counter() - start_enum) * 1000.0

    # 2. String Tabanlı Durum Kontrolü Benzetimi
    start_str = time.perf_counter()
    state_str = "TRACKING_STATE_LONG_NAME"
    for i in range(iterations):
        # String strcmp benzetimi:
        if state_str == "IDLE_STATE": val = 10
        elif state_str == "STANDBY_STATE": val = 20
        elif state_str == "TRACKING_STATE_LONG_NAME": val = 30
        else: val = 0
    t_str_ms = (time.perf_counter() - start_str) * 1000.0

    ratio = t_str_ms / max(t_enum_ms, 0.001)
    print(f"  -> enum class Switch [100.000 çağrı]: {t_enum_ms:.2f} ms")
    print(f"  -> String Durum Kontrolü [100.000 çağrı]: {t_str_ms:.2f} ms")
    print(f"  -> PERFORMANS FARKI: enum class {ratio:.1f}x kat daha hızlı ve deterministiktir!")
    return t_enum_ms, t_str_ms

if __name__ == "__main__":
    run_module_05_benchmark()


## 16. Modül 5: 4-Panelli Yayın Kalitesinde Dayanıklılık Dashboard Planı

Bu modülde üretilecek analitik sistem şeması (`gun_061_enum_class.png`), modern sistem mühendisliği standartlarında 4 stratejik paneli kapsar:

1. **Panel 1: Kapsam ve İsim Alanı İzolasyonu**: Global C-Enum isim sızıntısı ile `enum class` kapsüllenmiş mimarisinin karşılaştırması.
2. **Panel 2: Altyapı Tipi Bellek Tasarrufu**: `uint8_t`, `uint16_t`, `uint32_t` ve varsayılan `int` bellek boyutlarının ve RAM ayak izinin analizi.
3. **Panel 3: Tip Güvenliği ve Karşılaştırma Matrisi**: Farklı enum türlerinin karşılaştırılmasında C-Enum felaketi vs `enum class` güvenlik kalkanı.
4. **Panel 4: Benchmark Analizi**: 100.000 durum değerlendirmesinde `enum class` vs String kıyaslaması.


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# MODÜL 6: 4-Panelli Yayın Kalitesinde Dashboard ve Katı Doğrulama (Assertions)
# Matplotlib 300 DPI Dark Theme Dashboard (gun_061_enum_class.png)
# ==============================================================================

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

def generate_dashboard(output_path: str):
    fig, axes = plt.subplots(2, 2, figsize=(16, 11), dpi=300)
    fig.patch.set_facecolor('#0F172A')  # Slate 900 Koyu Arka Plan

    # --------------------------------------------------------------------------
    # Panel 1: Kapsam İzolasyonu
    # --------------------------------------------------------------------------
    ax1 = axes[0, 0]
    ax1.set_facecolor('#1E293B')
    ax1.set_title("1. Kapsam ve İsim Alanı İzolasyonu (Global vs Scoped)", color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

    # Eski C-Enum Kutusu
    b_c = patches.FancyBboxPatch((0.08, 0.52), 0.84, 0.40, boxstyle="round,pad=0.02", facecolor='#450A0A', edgecolor='#EF4444', linewidth=1.5)
    ax1.add_patch(b_c)
    ax1.text(0.12, 0.86, "ESKİ C-ENUM (Global Kapsama Sızma):", color='#FCA5A5', fontsize=9, fontweight='bold')
    ax1.text(0.12, 0.76, "enum RadarState { IDLE, ERROR };\nenum FlirState  { IDLE, ERROR }; // ÇAKIŞMA! DERLENMEZ!", color='#FFFFFF', fontsize=8, family='monospace')
    ax1.text(0.12, 0.58, "▲ İsim alanı kirlenir, farklı enum'lar birbirini ezer.", color='#F87171', fontsize=7.5)

    # Modern enum class Kutusu
    b_modern = patches.FancyBboxPatch((0.08, 0.08), 0.84, 0.38, boxstyle="round,pad=0.02", facecolor='#064E3B', edgecolor='#10B981', linewidth=1.5)
    ax1.add_patch(b_modern)
    ax1.text(0.12, 0.40, "MODERN C++ enum class (Kapsüllenmiş Ad Alanı):", color='#6EE7B7', fontsize=9, fontweight='bold')
    ax1.text(0.12, 0.30, "enum class RadarState { Idle, Error };\nenum class FlirState  { Idle, Error }; // TAMAMEN İZOLE!", color='#FFFFFF', fontsize=8, family='monospace')
    ax1.text(0.12, 0.14, "▲ RadarState::Idle ile FlirState::Idle bağımsızdır, çakışma %0!", color='#34D399', fontsize=7.5)

    ax1.set_xlim(0, 1); ax1.set_ylim(0, 1); ax1.axis('off')

    # --------------------------------------------------------------------------
    # Panel 2: Altyapı Tipi Bellek Ayak İzi
    # --------------------------------------------------------------------------
    ax2 = axes[0, 1]
    ax2.set_facecolor('#1E293B')
    ax2.set_title("2. Altyapı Tipi (Underlying Type) Bellek Tasarrufu", color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

    types = ['Varsayılan\n(int)', 'uint32_t', 'uint16_t', 'uint8_t\n(Tavsiye)']
    bytes_arr = [4, 4, 2, 1]
    colors_bar = ['#EF4444', '#F59E0B', '#3B82F6', '#10B981']

    bars = ax2.bar(types, bytes_arr, color=colors_bar, width=0.45, edgecolor='#38BDF8', linewidth=1.2)
    ax2.set_ylabel('Bayt Boyutu (sizeof)', color='#E2E8F0', fontsize=9)
    ax2.tick_params(axis='both', colors='#E2E8F0')
    ax2.set_ylim(0, 5)
    ax2.grid(axis='y', linestyle='--', alpha=0.2, color='#94A3B8')

    for bar in bars:
        h = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., h + 0.15, f"{h} Bayt",
                 ha='center', va='bottom', color='#FFFFFF', fontweight='bold', fontsize=9)

    ax2.text(0.50, 0.85, ": uint8_t Seçimi ile %75 Bellek Tasarrufu!\n(Cache Friendly & Bus Aligned)",
             transform=ax2.transAxes, color='#34D399', fontsize=8.5, ha='center', fontweight='bold',
             bbox=dict(boxstyle="round,pad=0.3", facecolor='#0F172A', edgecolor='#10B981', lw=1))

    # --------------------------------------------------------------------------
    # Panel 3: Tip Güvenliği ve Karşılaştırma Matrisi
    # --------------------------------------------------------------------------
    ax3 = axes[1, 0]
    ax3.set_facecolor('#1E293B')
    ax3.set_title("3. Güvenlik Bariyeri: C-Enum vs enum class", color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

    criteria = ['Kapsülleme', 'Örtülü İnt Yasağı', 'Sabit Boyut', 'Güvenli Bitmask']
    c_scores = [0, 0, 20, 30]
    modern_scores = [100, 100, 100, 100]

    x = np.arange(len(criteria))
    width = 0.35

    ax3.bar(x - width/2, c_scores, width, label='Eski C-Enum', color='#EF4444', alpha=0.85)
    ax3.bar(x + width/2, modern_scores, width, label='Modern enum class', color='#10B981', alpha=0.85)

    ax3.set_ylabel('Güvenlik Seviyesi (%)', color='#E2E8F0', fontsize=9)
    ax3.set_xticks(x)
    ax3.set_xticklabels(criteria, color='#E2E8F0', fontsize=8)
    ax3.set_ylim(0, 120)
    ax3.tick_params(axis='y', colors='#E2E8F0')
    ax3.grid(axis='y', linestyle='--', alpha=0.2, color='#94A3B8')
    ax3.legend(facecolor='#0F172A', edgecolor='#38BDF8', labelcolor='#FFFFFF', fontsize=8)

    # --------------------------------------------------------------------------
    # Panel 4: Benchmark Analizi
    # --------------------------------------------------------------------------
    ax4 = axes[1, 1]
    ax4.set_facecolor('#1E293B')
    ax4.set_title("4. 100.000 Durum Kontrolü (ms) [Düşük Olan İyidir]", color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

    bench_labels = ['String Durum\n(strcmp)', 'enum class\n(O(1) Jump Table)']
    bench_times = [9.4, 2.1]  # Temsili ampirik ölçüm
    colors = ['#F59E0B', '#10B981']

    bars2 = ax4.bar(bench_labels, bench_times, color=colors, width=0.45, edgecolor='#38BDF8', linewidth=1.2)
    ax4.set_ylabel('Süre (milisaniye)', color='#E2E8F0', fontsize=9)
    ax4.tick_params(axis='both', colors='#E2E8F0')
    ax4.set_ylim(0, max(bench_times) * 1.3)
    ax4.grid(axis='y', linestyle='--', alpha=0.2, color='#94A3B8')

    for bar in bars2:
        h = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., h + 0.3, f"{h:.1f} ms",
                 ha='center', va='bottom', color='#FFFFFF', fontweight='bold', fontsize=9)

    ax4.text(0.50, 0.85, "enum class ~4.5x Kat Daha Hızlı!\nDeterministik O(1) İşlemci Atlama Tablosu",
             transform=ax4.transAxes, color='#34D399', fontsize=8.5, ha='center', fontweight='bold',
             bbox=dict(boxstyle="round,pad=0.3", facecolor='#0F172A', edgecolor='#10B981', lw=1))

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, facecolor=fig.get_facecolor(), edgecolor='none')
    plt.close()
    print(f"[+] Görselleştirme Grafiği Üretildi: {output_path}")

def verify_all_assertions():
    print("=" * 60)
    print("GÜN 061: KATI DOĞRULAMA (ASSERTIONS) TESTLERİ BAŞLIYOR")
    print("=" * 60)

    class StateA(ScopedEnum): pass
    StateA.Init = StateA("Init", 0, 1)
    StateA.Run  = StateA("Run", 1, 1)

    class StateB(ScopedEnum): pass
    StateB.Init = StateB("Init", 0, 1)

    # 1. Test: Kapsam İçi Eşitlik ve Eşitsizlik
    assert StateA.Init == StateA.Init, "Aynı enum değeri eşit olmalıdır!"
    assert StateA.Init != StateA.Run, "Farklı enum değerleri eşit olmamalıdır!"
    print("  -> [PASS] Kapsam içi eşitlik ve eşitsizlik operatörleri doğrulandı.")

    # 2. Test: Farklı Enum Türlerinin Karşılaştırma Blokajı
    type_safety_passed = False
    try:
        _ = (StateA.Init == StateB.Init)
    except TypeError:
        type_safety_passed = True
    assert type_safety_passed is True, "Farklı enum türlerinin eşitliği TypeError fırlatmalıdır!"
    print("  -> [PASS] Farklı enum türleri arası örtülü karşılaştırma yasağı onaylandı.")

    # 3. Test: Boyut ve Altyapı Tipi Denetimi
    assert StateA.Init.sizeof == 1, "Altyapı tipi boyutu 1 bayt olmalıdır!"
    print("  -> [PASS] Altyapı tipi (uint8_t) bellek boyutu doğrulandı.")

    # 4. Test: Bitmask Operatörleri Doğrulaması
    f1 = BitmaskFlags(0x01)
    f2 = BitmaskFlags(0x02)
    f_comb = f1 | f2
    assert f_comb.underlying_mask == 0x03, "Bitwise OR sonucu 0x03 olmalıdır!"
    assert f_comb.contains(f1) is True, "Kombinasyon f1 bayrağını içermelidir!"
    print("  -> [PASS] Tip güvenli Bitmask operatörleri onaylandı.")

    print("=" * 60)
    print("TÜM ASSERT TESTLERİ BAŞARIYLA TAMAMLANDI! [PASS]")
    print("=" * 60)

if __name__ == "__main__":
    generate_dashboard("test_dashboard.png")
    verify_all_assertions()


## 18. ⚠️ 10 Ölümcül Anti-Pattern (enum class Kullanım Hataları)

| No | Anti-Pattern | Neden Tehlikeli? | Doğru Mühendislik Yaklaşımı |
| :--- | :--- | :--- | :--- |
| **1** | **Eski C-Tarzı `enum` Kullanmak** | Global kapsama sızar ve tamsayıya örtülü dönüşerek sistemleri karıştırır. | İstisnasız her zaman `enum class` kullanın. |
| **2** | **Altyapı Tipini Belirtmemek** | Varsayılan `int` (4 bayt) seçilir; donanım kayıtçısı ve ağ paketlerinde boyut uyuşmazlığı yaratır. | Ağ/Donanım kayıtçılarında daima `: uint8_t` gibi açık tip belirtin. |
| **3** | **Sınır Kontrolsüz `static_cast` Yapmak** | Ağdan gelen geçersiz bir tamsayı enum'a dönüştürülünce tanımsız durum oluşur. | Gelen baytı önce `isValid(byte)` fonksiyonundan geçirin. |
| **4** | **`switch` İçinde `default` Yazmamak** | Yeni bir enum eklendiğinde veya bozuk veri geldiğinde sistem askıda kalır. | Daima `default: throw / logError()` koruması ekleyin ve `-Wswitch` uyarısını açın. |
| **5** | **Enum Değerlerini Sihirli Sayı Gibi Kullanmak** | `state + 1` gibi tamsayı aritmetiği yapmak durum makinesini kırar. | Durum geçişlerini explicit `transition(CurrentState)` fonksiyonlarıyla yönetin. |
| **6** | **Global Kapsamda `using enum` Açmak** | Header dosyasında `using enum` yazmak tüm ad alanını kirletir. | `using enum` bildirimini sadece ilgili fonksiyon veya switch bloğu içinde yerel tutun. |
| **7** | **Aynı Anlama Gelen Yinelenen Enum'lar** | Farklı modüllerde benzer amaçlı onlarca enum kopyası bakım krizine yol açar. | Ortak durumları `CommonTypes.hpp` altında merkezi olarak konsolide edin. |
| **8** | **Bitmask İçin Düz `enum class` Bırakmak** | `operator|` aşırı yüklenmezse her bit işleminde çirkin `static_cast` yazmak gerekir. | Bitmask enum'ları için type-safe bitwise operatörlerini tanımlayın. |
| **9** | **Enum İsimlerine Ön Ek (Prefix) Eklemek** | `enum class State { STATE_IDLE };` yazmak `State::STATE_IDLE` gibi gereksiz tekrar yaratır. | Ad alanı zaten korunduğu için temiz isimler kullanın: `State::Idle`. |
| **10** | **Enum'ı Dizeye Çevirmek İçin Dinamik Bellek Kullanmak** | `std::string` döndürmek heap tahsisi yapar; aviyonik döngüyü yavaşlatır. | `std::string_view` döndüren `constexpr` fonksiyonlar yazın. |


## 19. 🔬 Sektörel Standartlar (ASELSAN, HAVELSAN, MISRA, DO-178C)

Savunma sanayiinde güvenilirlik en üst düzey mühendislik metriğidir:

### 19.1 MISRA C++:2023 Kural 10.1 & 10.2
- **Kural:** Geleneksel C-tarzı unscoped `enum` kullanımı kesinlikle **yasaktır**.
- **Zorunluluk:** Tüm numaralandırmalar `enum class` ile kapsam içine alınmalı ve altyapı tipi açıkça (`explicit underlying type`) belirtilmelidir.
- **Yaptırım:** Örtülü tamsayı dönüşümü içeren kodlar statik analizde (Coverity, Polyspace, SonarQube) doğrudan derleme hatası (*fatal error*) kabul edilir.

---

### 19.2 DO-178C DAL-A Uçuş Emniyet Standardı
- **Deterministik Durum Kapsamı:** Uçuş durum makinelerindeki tüm durumlar `enum class` ile modellenmeli ve `switch` bloklarında tüm olası değerler derleme zamanında tüketilmiş (*exhaustive switch*) olmalıdır.
- **Dış Veri Doğrulama:** Aviyonik veri yolundan (MIL-STD-1553) okunan komut kodları enum'a dönüştürülmeden önce yazılımsal sınır testine tabi tutulmalıdır.

---

### 19.3 C++ Core Guidelines
- **Enum.1:** *"Prefer enumerations over macros."*
- **Enum.2:** *"Use `enum class` over 'plain' `enum`."*
- **Enum.3:** *"Specify the underlying type of an enumeration only when necessary (e.g., for hardware mapping)."*
- **Enum.5:** *"Don't use `ALL_CAPS` for enumerators."*


## 20. 🛠️ Üretim Kontrol Listesi (Production Checklist - 12 Katı Kural)

Canlı askeri ve aviyonik projelere kod teslim etmeden önce aşağıdaki 12 kuralı tek tek doğrulayın:

- [ ] **1. Kural (No Plain Enum):** Kod tabanında tek bir düz `enum` anahtar kelimesi kalmadı, tümü `enum class` yapıldı mı?
- [ ] **2. Kural (Explicit Underlying Type):** Ağ ve donanım kayıtçısı enum'larında `: uint8_t` / `: uint16_t` açıkça yazıldı mı?
- [ ] **3. Kural (Size Assertion):** Donanım eşleme enum'ları `static_assert(sizeof(E) == N)` ile mühürlendi mi?
- [ ] **4. Kural (No Implicit Int):** Enum tamsayıya dönüştürülürken açık `static_cast` veya `to_underlying` kullanıldı mı?
- [ ] **5. Kural (Exhaustive Switch):** Switch bloklarında tüm enum üyeleri ele alındı veya `default` dalı eklendi mi?
- [ ] **6. Kural (Compiler Flags):** Derleyicide `-Wswitch -Wswitch-enum -Werror` bayrakları aktif mi?
- [ ] **7. Kural (Local Using Enum):** C++20 `using enum` bildirimleri sadece yerel fonksiyon/switch kapsamı içinde tutuldu mu?
- [ ] **8. Kural (Bitmask Operators):** Bit bayrakları için tip güvenli `operator|` ve `operator&` aşırı yüklendi mi?
- [ ] **9. Kural (Boundary Check):** Dış dünyadan gelen ham tamsayılar enum'a çevrilmeden önce aralık testinden geçti mi?
- [ ] **10. Kural (String Conversion):** Enum'dan dizeye dönüşümde `std::string_view` ile sıfır heap tahsisi yapıldı mı?
- [ ] **11. Kural (Naming Cleanliness):** Enum üyelerinde gereksiz prefix (ön ek) tekrarları (`State::STATE_IDLE`) temizlendi mi?
- [ ] **12. Kural (Static Analysis):** MISRA C++:2023 Rule 10.1 ihlali oluşturmadığı analiz aracıyla onaylandı mı?


## 21. ❓ Sıkça Sorulan Sorular (Mülakat & Saha Soruları)

### Soru 1: `enum class` ile `enum struct` arasında bir fark var mıdır?
**Kıdemli Sistem Mimarı Yanıtı:**
Hayır, C++ standardında `enum class` ile `enum struct` tamamen eş anlamlıdır (*exact synonyms*). Her ikisi de aynı kapsamlı ve güçlü tipli numaralandırmayı üretir. Ancak savunma ve aviyonik sektöründe endüstri standardı olarak `enum class` sözdizimi tercih edilir.

---

### Soru 2: `std::to_underlying` (C++23) nedir ve C++20'de bunu nasıl yazarız?
**Kıdemli Sistem Mimarı Yanıtı:**
`enum class`'ı altyapı tamsayısına dönüştürmek için sürekli `static_cast<std::underlying_type_t<E>>(val)` yazmak uzun ve hataya açıktır. C++23 bu işlemi standart kütüphaneye `std::to_underlying(e)` olarak eklemiştir.
C++20'de bu yardımcı fonksiyonu tek satırda yazabiliriz:
```cpp
template <typename E>
constexpr auto to_underlying(E e) noexcept {
    static_assert(std::is_enum_v<E>, "Sadece enum tipleri kabul edilir!");
    return static_cast<std::underlying_type_t<E>>(e);
}
```

---

### Soru 3: Bir enum class'a altyapı tipinin sınırları içinde ama tanımlanmamış bir tamsayı atanabilir mi?
**Kıdemli Sistem Mimarı Yanıtı:**
Evet! C++ standardına göre eğer altyapı tipi `uint8_t` ise, `static_cast<MyEnum>(250)` derlenir ve tanımsız davranış (*UB*) oluşturmaz (çünkü 250, `uint8_t` sınırları içindedir). Ancak enum tanımında 250 karşılığı bir sembol yoksa bu durum **mantıksal bir hatadır**. Bu yüzden aviyonik bus ayrıştırıcılarında dış baytlar enum'a cast edilmeden önce mutlaka bir doğrulama fonksiyonundan geçirilmelidir.


## 22. 🔮 Gelecek Perspektifi (Donanım & Mimari Trendleri)

### 1. C++26 Statik Yansıma (Static Reflection - P2996)
Günümüzde bir `enum class`'ı string'e çevirmek için yüzlerce satır `switch-case` veya makro yazmak zorundayız (`MagicEnum` kütüphanesi gibi şablon hileleri derleme süresini uzatır).
C++26 ile gelecek olan **Statik Yansıma (Reflection)** sayesinde:
```cpp
// C++26 Gelecek Vizyonu:
template <typename E>
constexpr std::string_view enum_to_string(E value) {
    constexpr auto enumerators = std::meta::enumerators_of(^E);
    // Derleme zamanında sıfır maliyetle string dönüşümü!
}
```
Bu özellik, savunma sanayiinde telemetri loglayıcıları ve seri haberleşme arayüzleri yazarken binlerce satırlık kod yükünü tamamen ortadan kaldıracaktır.

### 2. Donanım Seviyesinde Durum Doğrulama (Hardware-Enforced Enums)
Gelecek nesil askeri SoC'lerde (RISC-V Custom Instructions), donanımsal register yazma işlemlerinde izin verilmeyen enum değerleri doğrudan CPU seviyesinde *Hardware Trap* üreterek işletim sisteminin çekirdeğine arıza bildirimi yapacaktır.


## 23. 🚪 Sonraki Güne Köprü (Neden-Sonuç Bağı)

```text
[ GÜN 060: std::variant ] ── (Çoklu Tiplerden Biri: Tip Güvenli Union)
                 │
[ GÜN 061: enum class ] (BUGÜN) ── (Tipli Durum Sabitleri, Kapsüllenmiş İsim Alanı, Bitmask)
                 │
[ GÜN 062: Templates Temel Mantığı ] (YARIN) ── (Jenerik Tipler, Tip Parametrizasyonu, Sıfır Ek Maliyetli Kod Üretimi)
```

- **Bugün Ne Kazandık?**: Eski C-enum'larının yarattığı kapsam kirliliğini, örtülü tamsayı dönüşüm felaketlerini ve boyut belirsizliklerini yok eden `enum class`, belirlenmiş altyapı tipi (`underlying type`), tip güvenli bitmask ve C++20 `using enum` mimarisini mühürledik.
- **Yarın Ne Göreceğiz?**: Gün 062'de C++'ı C'den ve diğer dillerden ayıran en güçlü süper güce adım atıyoruz: **`Templates Temel Mantığı`**. Türleri fonksiyonlara ve sınıflara parametre olarak geçiren, çalışma zamanı maliyeti ödemeden (*Zero-Cost Abstraction*) derleme zamanında jenerik kod üreten meta-programlama evrenine giriş yapacağız!
